# AMBER — Beam Prediction on DeepSense6G (scenarios 31–34)End-to-end Colab notebook: **stage → (preprocess) → train → evaluate → predict**.Implements *"AMBER: An Adaptive Multimodal Mask Transformer for Beam Predictionwith Missing Modalities"* (Wen et al.) on the DeepSense6G 2022 Multi-ModalBeam Prediction dataset.---## Before you start — what must be in Google DriveYou need **the project code** and **the data**. The fast path is to preprocesson your own machine (already done) and upload only the processed tensors.**On your laptop**, create one archive and upload it to Drive:```bashcd "/Users/rakshith/Desktop/pe_5g:6g"tar -czf processed_amber.tar.gz -C data processed_amber      # ~8.6 GB -> a few GBtar -czf amber_code.tar.gz --exclude=__pycache__ amber preprocessing_amber requirements.txt```Put both in a Drive folder, e.g. `MyDrive/amber/`.> **Why an archive and not the raw folders?** The processed dataset is ~38,000> small files. Reading those one-by-one over Drive's FUSE mount is extremely> slow and will bottleneck training far more than the GPU. A single archive> extracted onto Colab's local disk avoids that entirely.If you would rather preprocess *in* Colab, upload `data/raw/` instead and set`RUN_PREPROCESSING = True` below — but note the raw data is **35 GB** andpreprocessing takes hours on a Colab CPU.## What the splits mean| split | n | usable for ||---|---|---|| `train` | 5,544 | fitting || `val` | 1,350 | model selection || `adaptation` | 100 | held-out labelled check || `test` | 625 | **unlabelled** — predictions only, no metrics |The official test release ships **no `mmWave_data`**, so beam history is absentfor 100 % of test samples but present for ~96 % of training ones. The notebooksets `--beam-dropout` high to stop the model depending on a signal it will nothave at inference. See the training section.

## 1. Check the GPURuntime ▸ Change runtime type ▸ **T4 GPU** (or better) before running.

In [ ]:
!nvidia-smi || echo "No GPU detected — set Runtime > Change runtime type > T4 GPU"import torch, platformprint(f"\npython {platform.python_version()}  torch {torch.__version__}")print(f"cuda available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"device: {torch.cuda.get_device_name(0)}")    print(f"memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. DependenciesColab already ships torch/torchvision; these are the rest.

In [ ]:
!pip install -q numpy scipy pandas pillow tqdm joblibprint("dependencies ready")

## 3. ConfigurationEdit `DRIVE_ROOT` to wherever you put the archives. Everything else has asensible default.

In [ ]:
from pathlib import Path# ---- where your files live in Drive -----------------------------------------DRIVE_ROOT      = Path("/content/drive/MyDrive/amber")CODE_ARCHIVE    = DRIVE_ROOT / "amber_code.tar.gz"        # or None if using gitDATA_ARCHIVE    = DRIVE_ROOT / "processed_amber.tar.gz"   # preferredRAW_DATA_DIR    = DRIVE_ROOT / "raw"                      # only if preprocessing here# ---- local (fast) working directories on the Colab VM -----------------------WORK            = Path("/content/work")PROJECT         = WORK / "project"DATA            = WORK / "data"INDEX_DIR       = DATA / "processed_amber" / "index"# ---- run settings -----------------------------------------------------------RUN_NAME          = "amber-full"RUN_DIR           = DRIVE_ROOT / "runs"      # checkpoints go to Drive so a                                             # disconnect cannot lose themEPOCHS            = 20                       # AMBER Table IIBATCH_SIZE        = 16                       # AMBER Table IILR                = 1e-4                     # AMBER Table IIPOOL              = 4                        # VA = HAMODALITY_DROPOUT  = 0.15BEAM_DROPOUT      = 0.50                     # see the note in the headerWORKERS           = 2RUN_PREPROCESSING = False                    # True only if you uploaded raw datafor d in (WORK, PROJECT, DATA):    d.mkdir(parents=True, exist_ok=True)print(f"project -> {PROJECT}\ndata    -> {DATA}\nruns    -> {RUN_DIR}")

## 4. Mount Drive

In [ ]:
from google.colab import drivedrive.mount("/content/drive")assert DRIVE_ROOT.exists(), f"{DRIVE_ROOT} not found — check DRIVE_ROOT in the config cell"print(f"\ncontents of {DRIVE_ROOT}:")for p in sorted(DRIVE_ROOT.iterdir()):    size = f"{p.stat().st_size/1e9:.2f} GB" if p.is_file() else "dir"    print(f"  {p.name:<32} {size}")

## 5. Stage the codeExtracts `amber_code.tar.gz` onto the local disk. If you would rather pull fromGitHub, replace this cell with a `git clone` — just make sure the branch has the`amber/` package and `preprocessing_amber/`.

In [ ]:
import subprocess, sys, osif CODE_ARCHIVE and CODE_ARCHIVE.exists():    subprocess.run(["tar", "-xzf", str(CODE_ARCHIVE), "-C", str(PROJECT)], check=True)    print(f"extracted {CODE_ARCHIVE.name}")else:    # Alternative: pull the code from GitHub instead of Drive.    # !git clone https://github.com/SathishAdithiyaaSV/MultiModal-Beam-Prediction.git {PROJECT}    raise FileNotFoundError(        f"{CODE_ARCHIVE} not found. Upload amber_code.tar.gz, or switch this cell "        f"to git clone.")os.chdir(PROJECT)sys.path.insert(0, str(PROJECT))print(f"\ncwd = {Path.cwd()}")print("contents:", sorted(p.name for p in PROJECT.iterdir()))

## 6. Stage the dataExtracting to Colab's local disk rather than reading from Drive is the singlebiggest speed win in this notebook — see the note in the header.

In [ ]:
import shutil, timeif INDEX_DIR.exists():    print(f"data already staged at {DATA/'processed_amber'}")elif DATA_ARCHIVE.exists():    t0 = time.time()    print(f"extracting {DATA_ARCHIVE.name} ({DATA_ARCHIVE.stat().st_size/1e9:.2f} GB) ...")    subprocess.run(["tar", "-xzf", str(DATA_ARCHIVE), "-C", str(DATA)], check=True)    print(f"done in {time.time()-t0:.0f}s")elif RUN_PREPROCESSING and RAW_DATA_DIR.exists():    print("no processed archive; will preprocess from raw in the next cell")    (DATA / "raw").mkdir(parents=True, exist_ok=True)    print(f"linking raw data from {RAW_DATA_DIR}")    for scn_src in sorted(RAW_DATA_DIR.iterdir()):        dst = DATA / "raw" / scn_src.name        if not dst.exists():            dst.symlink_to(scn_src)else:    raise FileNotFoundError(        "Neither a processed archive nor raw data found. Upload "        "processed_amber.tar.gz, or set RUN_PREPROCESSING=True with raw data in Drive.")if INDEX_DIR.exists():    print("\nindex files:")    for p in sorted(INDEX_DIR.iterdir()):        print(f"  {p.name:<24} {p.stat().st_size/1e6:.1f} MB")

## 7. Preprocessing *(only if you uploaded raw data)*Skipped automatically when the processed index is already present. The fourstages are radar → LiDAR BEV → camera cache → index, each idempotent, so are-run only does missing work.

In [ ]:
if RUN_PREPROCESSING and not INDEX_DIR.exists():    # point the pipeline's config at the staged data directory    os.environ["AMBER_DATA_ROOT"] = str(DATA)    stages = [        ("audit",  ["preprocessing_amber/audit_dataset.py", "--quick"]),        ("radar",  ["preprocessing_amber/radar_ra_rv.py", "--source", "all"]),        ("lidar",  ["preprocessing_amber/lidar_bev.py", "--source", "all"]),        ("camera", ["preprocessing_amber/image_cache.py", "--source", "all"]),        ("index",  ["preprocessing_amber/build_index.py"]),    ]    for name, cmd in stages:        print(f"\n{'='*70}\n{name}\n{'='*70}")        subprocess.run([sys.executable, *cmd], check=(name != "audit"))else:    print("preprocessing skipped — processed index already present")

## 8. Sanity-check the indexConfirms the splits and modality availability before spending GPU hours.

In [ ]:
import pandas as pd, jsonsamples = pd.read_csv(INDEX_DIR / "samples.csv")meta    = json.loads((INDEX_DIR / "meta.json").read_text())print(f"{len(samples)} rows, {len(samples.columns)} columns\n")print("split sizes:")print(samples.split.value_counts().to_string(), "\n")print("modality availability (1.0 = always present):")usable = samples[samples.split.isin(["train", "val", "adaptation", "test"])]print(usable.groupby("split")[["m_image","m_lidar","m_radar","m_beam","m_gps"]]      .mean().round(3).to_string(), "\n")print("per-scenario counts:")print(usable.groupby(["split","scenario"]).size().to_string(), "\n")print("split policy:", json.dumps(meta["split_policy"], indent=2))assert (samples.split == "train").sum() > 0, "no training rows — staging failed"print("\nindex looks good")

## 9. TrainAMBER's Table II settings: batch 16, lr 1e-4, 20 epochs, AdamW, cosine schedulewith 5 warm-up steps, 8 heads, dropout 0.1, losses weighted 10 / 0.2 / 0.2.**`--beam-dropout 0.5`** is not from the paper. It is here because the officialtest release contains no `mmWave_data`, so beam history — which AMBER's ownTable V shows is its strongest cheap signal (BeamIdx+GPS alone reaches 58.81 %Top-1) — is missing for every test sample. Training with the paper's uniform0.15 would produce a model that leans on it and then degrades sharply atinference. Set it to `MODALITY_DROPOUT` if you want the paper's exact recipe.Checkpoints land in Drive, and `--resume` continues from `last.pt`, so**re-running this cell after a Colab disconnect picks up where it stopped.**Expect roughly 10–25 min per epoch on a T4.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)cmd = [    sys.executable, "-m", "amber.train",    "--index-dir", str(INDEX_DIR),    "--run-dir",   str(RUN_DIR),    "--name",      RUN_NAME,    "--epochs",    str(EPOCHS),    "--batch-size", str(BATCH_SIZE),    "--lr",        str(LR),    "--pool",      str(POOL),    "--modality-dropout", str(MODALITY_DROPOUT),    "--beam-dropout",     str(BEAM_DROPOUT),    "--workers",   str(WORKERS),    "--device",    "cuda",    "--resume",]print(" ".join(cmd), "\n")!{" ".join(cmd)}

## 10. Training curves

In [ ]:
import matplotlib.pyplot as plthistory = pd.read_csv(RUN_DIR / RUN_NAME / "history.csv")fig, ax = plt.subplots(1, 3, figsize=(15, 4))ax[0].plot(history.epoch, history.loss, label="total")ax[0].plot(history.epoch, history.focal, label="focal")ax[0].plot(history.epoch, history.contrastive, label="contrastive")ax[0].set(xlabel="epoch", ylabel="loss", title="Training loss")ax[0].legend(); ax[0].grid(alpha=.3)ax[1].plot(history.epoch, history.val_top1, marker="o", label="Top-1")ax[1].plot(history.epoch, history.val_top3, marker="s", label="Top-3")ax[1].set(xlabel="epoch", ylabel="accuracy", title="Validation accuracy")ax[1].legend(); ax[1].grid(alpha=.3)ax[2].plot(history.epoch, history.val_dba, marker="^", color="tab:green")ax[2].set(xlabel="epoch", ylabel="DBA", title="Validation DBA score")ax[2].grid(alpha=.3)plt.tight_layout(); plt.show()print(history.tail(5).to_string(index=False))print(f"\nbest val Top-1: {history.val_top1.max():.4f} "      f"at epoch {int(history.loc[history.val_top1.idxmax(),'epoch'])}")

## 11. Evaluate the best checkpointScores `val` and `adaptation` — the two splits that have labels — broken downper scenario, using Top-1 / Top-3 / Top-5 and the DBA score of eqs. (37)–(39).

In [ ]:
CKPT = RUN_DIR / RUN_NAME / "best.pt"assert CKPT.exists(), f"{CKPT} not found — did training complete an epoch?"for split in ("val", "adaptation"):    print(f"\n{'='*70}\n{split}\n{'='*70}")    !{sys.executable} -m amber.predict \        --checkpoint "{CKPT}" --index-dir "{INDEX_DIR}" --split {split} \        --out-dir "{RUN_DIR / RUN_NAME}" --batch-size 32 --workers {WORKERS} --device cuda

## 12. Predict on the official test setThe test release is **unlabelled**, so this produces predictions rather thanmetrics. Two files are written:- `predictions_test.csv` — ranked top-5 beams plus confidences, per sample- `submission_test.csv` — the top-1 beam only, 1-indexed, in the official  CSV's row orderBeam indices are written **1-indexed** to match the dataset's `unit1_beam`convention, while the model works 0-indexed internally.

In [ ]:
!{sys.executable} -m amber.predict \    --checkpoint "{CKPT}" --index-dir "{INDEX_DIR}" --split test \    --out-dir "{RUN_DIR / RUN_NAME}" --batch-size 32 --workers {WORKERS} --device cudapreds = pd.read_csv(RUN_DIR / RUN_NAME / "predictions_test.csv")print(f"\n{len(preds)} test predictions")print(preds.head(10).to_string(index=False))fig, ax = plt.subplots(1, 2, figsize=(13, 4))ax[0].hist(preds.top1_beam, bins=64, range=(1, 65), color="tab:blue")ax[0].set(xlabel="predicted beam (1-indexed)", ylabel="count",          title="Predicted beam distribution")ax[0].grid(alpha=.3)preds.groupby("scenario").size().plot.bar(ax=ax[1], color="tab:orange", rot=0)ax[1].set(ylabel="samples", title="Test samples per scenario")ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()n_distinct = preds.top1_beam.nunique()print(f"\ndistinct beams predicted: {n_distinct}/64")if n_distinct < 5:    print("WARNING: the model is predicting almost one beam for everything — "          "it has not learned. Check the training curves above.")

## 13. Everything is already in DriveCheckpoints, metrics and predictions were written straight to `RUN_DIR`.

In [ ]:
out = RUN_DIR / RUN_NAMEprint(f"{out}:\n")for p in sorted(out.iterdir()):    print(f"  {p.name:<30} {p.stat().st_size/1e6:>9.2f} MB")print("\nfiles you want to keep:")print("  best.pt                  best-val checkpoint")print("  history.csv              per-epoch training log")print("  metrics_val.json         validation metrics, per scenario")print("  metrics_adaptation.json  held-out metrics, per scenario")print("  predictions_test.csv     ranked top-5 beams for the official test set")print("  submission_test.csv      top-1 beam only")

## Troubleshooting**Colab disconnected mid-training.** Re-run the training cell. `--resume` reads`last.pt` from Drive and continues from the next epoch.**Training is very slow per epoch.** Confirm the data was extracted to`/content/work/data` and not being read from `/content/drive`. Drive's FUSEmount makes 38,000 small reads per epoch painfully slow.**Out of memory.** Lower `BATCH_SIZE` to 8, or `POOL` to 2 (fewer tokens permodality). `POOL=4` gives 16 tokens per grid modality; `POOL=2` gives 4.**Model predicts one beam for everything.** Normal for the first couple ofepochs. If it persists past ~5 epochs, the loss weighting or learning rate isoff — check that `focal` in the training log is decreasing.**Want the paper's exact recipe.** Set `BEAM_DROPOUT = MODALITY_DROPOUT` (0.15).Validation Top-1 will look better and test predictions will likely be worse,for the reason explained in the training section.## Known data limitations- **Scenario 34 has no labels in the development set** — the release ships no  radar, vehicle GPS or power files, so its 4,191 samples sit in `no_target`  and cannot be trained on. Only scenarios 32 and 33 contribute to `train`/`val`.- **Scenario 31 has no training data at all** — it appears only in the  100-sample `adaptation` set and the unlabelled `test` set.- **58 development samples have corrupt labels** and are quarantined in  `excluded_nan_pwr`.So per-scenario results are available for 32 and 33 from `val`, and for 31/32/33from `adaptation`. The paper's per-scenario S31/S34 numbers need the fullper-scenario DeepSense6G releases, which are a separate download.